# Week 60, Day 3: How many clusters? BIC, the gap statistic and stability

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w060/d3.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def kmeans(X, k, g, restarts=3, iters=60):
    best = None
    for _ in range(restarts):
        C = X[g.choice(len(X), k, replace=False)]
        lab = None
        for _ in range(iters):
            new = np.argmin(((X[:, None] - C[None]) ** 2).sum(-1), 1)
            if lab is not None and np.array_equal(new, lab): break
            lab = new
            C = np.array([X[lab == j].mean(0) if np.any(lab == j) else C[j] for j in range(k)])
        W = ((X - C[lab]) ** 2).sum()
        if best is None or W < best[1]:
            best = (lab, W, C)
    return best

g0 = np.random.default_rng(603)
true_c = np.array([[0, 0], [4, 0], [0, 4], [4, 4.5]])
X4 = np.repeat(true_c, 75, axis=0) + 0.8 * g0.normal(size=(300, 2))          # four groups
X1 = g0.normal(size=(300, 2)) * [1.8, 1.2]                                    # one cloud, no groups
for name, X in [("four groups", X4), ("no groups", X1)]:
    W = [kmeans(X, k, g0)[1] for k in range(1, 9)]
    plt.plot(range(1, 9), W / W[0], "o-", label=name)
plt.xlabel("k"); plt.ylabel("distortion (relative to k = 1)"); plt.legend(); plt.show()

### Exercise: BIC for Gaussian mixtures

Write n_params(K, d) and bic(X, K, g), which fits a mixture with gmm_fit (best of two starts) and returns -2 log-likelihood plus n_params times log n. Choose K from 1 to 7 for both datasets.

In [ ]:
def mvn_pdf(X, m, S):
    L = np.linalg.cholesky(S); Z = np.linalg.solve(L, (X - m).T)
    return np.exp(-0.5 * (Z ** 2).sum(0) - np.log(np.diag(L)).sum() - 0.5 * len(m) * np.log(2 * np.pi))

def gmm_fit(X, K, g, iters=150, reg=1e-4, tol=1e-4):
    n, d = X.shape
    mu = X[g.choice(n, K, replace=False)]; Sig = np.array([np.cov(X.T)] * K); pi = np.full(K, 1 / K)
    prev = -np.inf
    for _ in range(iters):
        dens = np.column_stack([pi[k] * mvn_pdf(X, mu[k], Sig[k]) for k in range(K)]) + 1e-300
        ll = np.log(dens.sum(1)).sum()
        if ll - prev < tol: break
        prev = ll
        R = dens / dens.sum(1, keepdims=True); Nk = R.sum(0) + 1e-9
        pi, mu = Nk / n, (R.T @ X) / Nk[:, None]
        Sig = np.array([((R[:, k, None] * (X - mu[k])).T @ (X - mu[k])) / Nk[k] + reg * np.eye(d) for k in range(K)])
    dens = np.column_stack([pi[k] * mvn_pdf(X, mu[k], Sig[k]) for k in range(K)])
    return np.log(dens.sum(1) + 1e-300).sum()

def n_params(K, d):
    return ...

def bic(X, K, g):
    ll = max(gmm_fit(X, K, g) for _ in range(2))
    return ...

gb = np.random.default_rng(6031)
bic4 = [bic(X4, K, gb) for K in range(1, 8)]
bic1 = [bic(X1, K, gb) for K in range(1, 8)]
k_bic_4, k_bic_1 = int(np.argmin(bic4)) + 1, int(np.argmin(bic1)) + 1
print("four groups, BIC:", np.round(bic4, 0), "->", k_bic_4)
print("one cloud, BIC:  ", np.round(bic1, 0), "->", k_bic_1)

In [ ]:
import numpy as np
def test_params():
    "n_params counts weights, means and full covariances"
    assert n_params(3, 2) == 2 + 6 + 9
def test_choice():
    "BIC chooses 4 components for the four groups, and 1 for the single cloud"
    assert k_bic_4 == 4 and k_bic_1 == 1

test_params()
test_choice()
print("All checks passed")

### Exercise: The gap statistic

Write gap_statistic(X, kmax, B, g): for k = 1 to kmax, compute log W_k on the data and on B uniform reference sets drawn in the bounding box, and return the gaps and the inflated standard deviations. Then apply the decision rule to both datasets.

In [ ]:
def gap_statistic(X, kmax, B, g):
    lo, hi = X.min(0), X.max(0)
    logW = np.array([np.log(kmeans(X, k, g)[1]) for k in range(1, kmax + 1)])
    ref = np.array([[np.log(kmeans(g.uniform(lo, hi, X.shape), k, g, restarts=2)[1]) for k in range(1, kmax + 1)]
                    for _ in range(B)])
    gap = ...
    s = ref.std(0) * np.sqrt(1 + 1 / B)
    return gap, s

def choose_k(gap, s):
    for k in range(len(gap) - 1):
        if ...:
            return k + 1
    return len(gap)

gg = np.random.default_rng(6032)
gap4, s4 = gap_statistic(X4, 7, 10, gg)
gap1, s1 = gap_statistic(X1, 7, 10, gg)
k_gap_4, k_gap_1 = choose_k(gap4, s4), choose_k(gap1, s1)
print("four groups, gap:", np.round(gap4, 2), "->", k_gap_4)
print("one cloud, gap:  ", np.round(gap1, 2), "->", k_gap_1)

In [ ]:
import numpy as np
def test_choice():
    "the gap rule chooses 4 for the four groups and 1 for the single cloud"
    assert k_gap_4 == 4 and k_gap_1 == 1

test_choice()
print("All checks passed")

### Exercise: Stability under resampling

Write pair_agreement(a, b), the fraction of pairs of points on which two labelings agree about being together or apart. Then, for k from 2 to 6, average it over 10 pairs of random half-samples, labelling all points by each half's nearest centres.

In [ ]:
def pair_agreement(a, b):
    same_a = a[:, None] == a[None, :]
    same_b = b[:, None] == b[None, :]
    iu = np.triu_indices(len(a), 1)
    return ...

def stability(X, k, g, reps=10):
    vals = []
    for _ in range(reps):
        perm = g.permutation(len(X)); h1, h2 = perm[: len(X) // 2], perm[len(X) // 2:]
        C1, C2 = kmeans(X[h1], k, g)[2], kmeans(X[h2], k, g)[2]
        l1 = np.argmin(((X[:, None] - C1[None]) ** 2).sum(-1), 1)
        l2 = np.argmin(((X[:, None] - C2[None]) ** 2).sum(-1), 1)
        vals.append(pair_agreement(l1, l2))
    return np.mean(vals)

gst = np.random.default_rng(6033)
stab4 = {k: stability(X4, k, gst) for k in range(2, 7)}
stab1 = {k: stability(X1, k, gst) for k in range(2, 7)}
print("four groups:", {k: round(float(v), 3) for k, v in stab4.items()})
print("one cloud:  ", {k: round(float(v), 3) for k, v in stab1.items()})

In [ ]:
import numpy as np
def test_pair():
    "pair_agreement is 1 for identical labelings up to renaming"
    assert pair_agreement(np.array([0, 0, 1, 1]), np.array([1, 1, 0, 0])) == 1.0
def test_four():
    "for the four groups, k = 4 is more stable than k = 3 and k = 5"
    assert stab4[4] > stab4[3] and stab4[4] > stab4[5]

test_pair()
test_four()
print("All checks passed")